##Ejercicio 3


In [ ]:
import time
import numpy as np


class Transform3D:
    """Clase modular para representar y operar transformaciones homogéneas en SE(3)."""

    def __init__(
        self,
        R: np.ndarray = None,
        p: np.ndarray = None,
        T: np.ndarray = None,
    ):
        if T is not None:
            self.T = np.array(T, dtype=np.float64)
        else:
            self.T = np.eye(4, dtype=np.float64)
            if R is not None:
                self.T[:3, :3] = R
            if p is not None:
                self.T[:3, 3] = p

    @property
    def R(self) -> np.ndarray:
        return self.T[:3, :3]

    @property
    def p(self) -> np.ndarray:
        return self.T[:3, 3]

    def inv_analytic(self) -> "Transform3D":
        """Calcula la inversa analítica rápida utilizando [-R^T * p]."""
        R_T = self.R.T
        p_inv = -R_T @ self.p
        T_inv = np.eye(4, dtype=np.float64)
        T_inv[:3, :3] = R_T
        T_inv[:3, 3] = p_inv
        return Transform3D(T=T_inv)

    def inv_generic(self) -> "Transform3D":
        """Calcula la inversa empleando la función genérica de NumPy."""
        return Transform3D(T=np.linalg.inv(self.T))

# EJECUCIÓN Y VALIDACIÓN NUMÉRICA
# 1. Definición de la matriz del Ejercicio 1
th, phi = np.radians(60), np.radians(90)
Rz = np.array(
    [[np.cos(th), -np.sin(th), 0], [np.sin(th), np.cos(th), 0], [0, 0, 1]]
)
Rx = np.array(
    [[1, 0, 0], [0, np.cos(phi), -np.sin(phi)], [0, np.sin(phi), np.cos(phi)]]
)
R_ej1 = Rz @ Rx
p_ej1 = Rz @ np.array([0.4, -0.2, 0.1])

T_ej1 = Transform3D(R=R_ej1, p=p_ej1)

# 2. Validación de Ortogonalidad / Error (Norma de Frobenius)
T_inv_analitica = T_ej1.inv_analytic()
E = T_ej1.T @ T_inv_analitica.T - np.eye(4)
frob_norm = np.linalg.norm(E, ord="fro")

print("=== VALIDACIÓN NUMÉRICA ===")
print(f"Norma de Frobenius ||T * T^-1 - I||_F : {frob_norm:.2e}")
print(f"¿Cumple ||E||_F < 1e-14? : {frob_norm < 1e-14}\n")

# 3. Benchmark Temporal (100,000 iteraciones)
N = 100_000

t0 = time.perf_counter()
for _ in range(N):
    _ = T_ej1.inv_analytic()
t1 = time.perf_counter()
t_analitico = t1 - t0

t0 = time.perf_counter()
for _ in range(N):
    _ = T_ej1.inv_generic()
t1 = time.perf_counter()
t_generico = t1 - t0

reduccion_pct = ((t_generico - t_analitico) / t_generico) * 100

print(f"=== BENCHMARK TEMPORAL ({N:,} iteraciones) ===")
print(f"Tiempo Inversa Analítica  : {t_analitico:.4f} s")
print(f"Tiempo Inversa Genérica   : {t_generico:.4f} s")
print(f"Reducción de Tiempo       : {reduccion_pct:.2f}%")

=== VALIDACIÓN NUMÉRICA ===
Norma de Frobenius ||T * T^-1 - I||_F : 2.10e-17
¿Cumple ||E||_F < 1e-14? : True

=== BENCHMARK TEMPORAL (100,000 iteraciones) ===
Tiempo Inversa Analítica  : 1.7460 s
Tiempo Inversa Genérica   : 1.5230 s
Reducción de Tiempo       : -14.65%
